# 문제 2-3 : PydanticOutputParser 활용 - 학생 정보 구조화 시스템

학생의 자유 형식 자기소개에서 이름, 나이, 전공, 취미 리스트, 목표를 추출해 구조화된 객체로 만듭니다.

- `PydanticOutputParser`와 `BaseModel` 사용
- 각 필드에 적절한 타입과 설명 추가
- 자유 형식의 텍스트에서 정보 추출

## 1. 환경 설정

In [3]:
from dotenv import load_dotenv
load_dotenv()

import os
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
GROQ_API_KEY = os.getenv("GROQ_API_KEY")
print("OpenAI Key:", "OK" if OPENAI_API_KEY else "없음", "| Groq Key:", "OK" if GROQ_API_KEY else "없음")

OpenAI Key: OK | Groq Key: OK


### LLM 제공자 선택

`PROVIDER` 값만 `"groq"` 또는 `"openai"`로 바꾸면 이후 모든 셀의 모델이 바뀝니다.
Groq는 OpenAI 호환 API라서 둘 다 `ChatOpenAI` 클래스를 사용하고 `base_url`만 다릅니다.

In [4]:
from langchain_openai import ChatOpenAI

# 사용할 LLM 제공자 선택: "groq" 또는 "openai" — 이 값만 바꾸면 이후 모든 셀의 모델이 바뀜
PROVIDER = "groq"

# 제공자별 설정 (Groq는 OpenAI 호환 API라서 base_url만 다르고, OpenAI는 base_url 생략 시 기본 엔드포인트 사용)
LLM_CONFIGS = {
    "groq": {
        "api_key": GROQ_API_KEY,
        "base_url": "https://api.groq.com/openai/v1",  # Groq API 엔드포인트
        "model": "openai/gpt-oss-120b",
    },
    "openai": {
        "api_key": OPENAI_API_KEY,
        "base_url": None,
        "model": "gpt-4o-mini",  # 대안: "gpt-4o"
    },
}
MAX_RETRIES = 5  # 503 등 일시 장애 자동 재시도 횟수


def get_llm(temperature: float = 0.5, provider: str | None = None, verbose: bool = True, **kwargs) -> ChatOpenAI:
    """선택한 제공자(기본값: PROVIDER)의 ChatOpenAI 인스턴스를 생성합니다. verbose=True 이면 생성된 모델 정보를 출력합니다.
    kwargs: ChatOpenAI 에 그대로 전달할 추가 옵션 (예: timeout=60)"""
    provider = provider or PROVIDER
    if provider not in LLM_CONFIGS:
        raise ValueError(f"지원하지 않는 제공자: {provider} (사용 가능: {list(LLM_CONFIGS)})")
    config = LLM_CONFIGS[provider]
    if not config["api_key"]:
        raise RuntimeError(f".env 파일에 {provider.upper()}_API_KEY 가 없습니다.")
    llm = ChatOpenAI(
        **config,
        temperature=temperature,
        max_retries=MAX_RETRIES,
        **kwargs,
    )
    if verbose:
        # 생성된 모델 정보 출력 (API 키는 출력하지 않음)
        print(f"[LLM 생성] provider={provider} | model={llm.model_name}")
    return llm

In [5]:
import json
from pydantic import BaseModel, Field
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.exceptions import OutputParserException

llm = get_llm(temperature=0)

[LLM 생성] provider=groq | model=openai/gpt-oss-120b


## 2. 스키마 정의 (BaseModel)

- `Field(description=...)`의 설명은 JSON 스키마로 바뀌어 형식 지침에 들어갑니다. 즉 **description이 모델에게 주는 추출 규칙**입니다.
- 파싱 후 Pydantic이 **타입을 검증**합니다. `age`에 `"스물두 살"`이 오면 실패하므로, 숫자로 변환하라는 규칙을 description에 적어 둡니다.

In [6]:
class StudentInfo(BaseModel):
    name: str = Field(description="학생의 이름")
    age: int = Field(description="나이. 한글 표현(스물두 살)도 숫자로 변환")
    major: str = Field(description="전공 분야 (예: 컴퓨터공학)")
    hobbies: list[str] = Field(description="취미 목록. 각 취미를 짧은 명사구로 (예: 게임하기)")
    goal: str = Field(description="앞으로의 목표를 '~하는 것' 형태로 요약")


output_parser = PydanticOutputParser(pydantic_object=StudentInfo)
format_instructions = output_parser.get_format_instructions()
print(format_instructions)

The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}
the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output schema:
```
{"properties": {"name": {"description": "학생의 이름", "title": "Name", "type": "string"}, "age": {"description": "나이. 한글 표현(스물두 살)도 숫자로 변환", "title": "Age", "type": "integer"}, "major": {"description": "전공 분야 (예: 컴퓨터공학)", "title": "Major", "type": "string"}, "hobbies": {"description": "취미 목록. 각 취미를 짧은 명사구로 (예: 게임하기)", "items": {"type": "string"}, "title": "Hobbies", "type": "array"}, "goal": {"description": "앞으로의 목표를 '~하는 것' 형태로 요약", "title": "Goal", "type": "string"}}, "required": ["name", "age", "major", "hobbies", "goal"]}
```


## 3. 프롬프트 + LCEL 체인

In [7]:
prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 자기소개 글에서 학생 정보를 추출하는 도우미입니다. 글에 있는 내용만 사용하세요.\n{format_instructions}"),
    ("user", "{intro}"),
]).partial(format_instructions=format_instructions)

chain = prompt | llm | output_parser

## 4. 실행

결과는 dict가 아니라 `StudentInfo` 객체입니다. 속성(`student.age`)으로 접근하고, JSON으로 보려면 `model_dump()`를 씁니다.

In [8]:
intro = "안녕하세요! 저는 김민수이고 22살입니다. 컴퓨터공학을 전공하고 있어요. 취미로는 게임하기, 영화보기, 코딩을 좋아합니다. 앞으로 훌륭한 개발자가 되는 것이 목표입니다."

student = chain.invoke({"intro": intro})
print(type(student).__name__)
print(f"이름: {student.name} / 나이: {student.age} ({type(student.age).__name__})")
print(json.dumps(student.model_dump(), indent=4, ensure_ascii=False))

StudentInfo
이름: 김민수 / 나이: 22 (int)
{
    "name": "김민수",
    "age": 22,
    "major": "컴퓨터공학",
    "hobbies": [
        "게임하기",
        "영화보기",
        "코딩"
    ],
    "goal": "훌륭한 개발자가 되는 것"
}


In [9]:
intros = [
    "반갑습니다~ 이서연이라고 해요. 올해 스물네 살이고 경영학과 다니고 있어요. 주말엔 등산 가거나 사진 찍으러 다니는 걸 좋아해요. 나중에 제 이름 건 카페를 차리는 게 꿈이에요!",
    "박준호, 20세, 전자공학 전공입니다. 기타 연주랑 축구를 즐깁니다. 졸업 전에 반도체 회사 인턴을 해보고 싶습니다.",
]

for text in intros:
    try:
        s = chain.invoke({"intro": text})
        print(json.dumps(s.model_dump(), indent=4, ensure_ascii=False))
    except OutputParserException as e:
        print(f"파싱 오류: {str(e).splitlines()[0]}")
    print("-" * 50)

{
    "name": "이서연",
    "age": 24,
    "major": "경영학",
    "hobbies": [
        "등산",
        "사진 찍기"
    ],
    "goal": "이름이 적힌 카페를 차리는 것"
}
--------------------------------------------------
{
    "name": "박준호",
    "age": 20,
    "major": "전자공학",
    "hobbies": [
        "기타 연주",
        "축구"
    ],
    "goal": "반도체 회사 인턴을 하는 것"
}
--------------------------------------------------
